# 03 · Storage effect in drought years (Iowa corn and soybeans)

**Direction:** Soil and Crop Resilience Planning. **User:** Soil Conservation Office.
**Question:** in drought years, do Iowa counties with more soil water storage (`aws_100cm_mm`) lose less yield relative to their own trend?

**This is an in-sample estimate.** The outcome, `yield_anomaly_pct`, is a residual from a county trend fitted on all of 2000–2025 (see the data dictionary), so every year, including the years analysed here, helped define "normal". The held-out test on years not used to build the estimate comes in a later notebook.

**Method, per crop:**
1. Keep Iowa county-years with a reported yield and `max_drought_severity >= 3` (D2+).
2. Within each year, use only that year's D2+ counties. Drop years with fewer than `MIN_COUNTIES_PER_YEAR` such counties.
3. For each remaining year, regress `yield_anomaly_pct` on `aws_100cm_mm` across that year's D2+ counties; express the slope as percentage points of yield anomaly per **+10 mm** of storage.
4. Overall estimate = average of the per-year slopes. 95% range from a **bootstrap over years** (resample years with replacement, recompute the average). Counties are never resampled, because counties in the same year share the same weather.
5. Four specifications: **A** storage only; **B** + `weeks_in_d2_plus` + `extreme_heat_days`; **C** + county centroid longitude (region); **D** + `nccpi_crop`.
6. Plain-language version: within each year, mean anomaly of high-storage minus low-storage counties (state-wide top vs bottom third by storage, among that year's D2+ counties), averaged over years, with the same year bootstrap.

Storage is a static county attribute, so everything here is an **association across counties**, not the measured effect of any program.

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from src.data_loader import load_master, load_counties

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)

df = load_master()
print(f"{len(df):,} rows × {len(df.columns)} columns")

17,056 rows × 21 columns


In [2]:
# ---- Column names (master dataset) ----
COL_FIPS = "fips"
COL_COUNTY = "county_name"
COL_YEAR = "year"
COL_STATE = "state_alpha"
COL_CROP = "crop"
COL_YIELD = "yield_per_acre"
COL_ANOM = "yield_anomaly_pct"          # outcome
COL_SEV = "max_drought_severity"        # 0-5; 3 = D2, 4 = D3, 5 = D4
COL_WEEKS_D2 = "weeks_in_d2_plus"
COL_HEAT = "extreme_heat_days"
COL_AWS = "aws_100cm_mm"                # exposure of interest
COL_NCCPI = "nccpi_crop"
COL_LON = "lon"                         # county centroid longitude, computed from counties.geojson

# ---- Drought definitions ----
SEV_D2 = 3
SEV_D3 = 4
MAIN_SEV = SEV_D2                        # main definition: D2+
SENS_SEV = SEV_D3                        # sensitivity: D3+

# ---- Scope and rules ----
FOCUS_STATE = "IA"
FOCUS_CROPS = ["CORN", "SOYBEANS"]
MIN_COUNTIES_PER_YEAR = 15
AWS_UNIT = 10.0                          # report slopes per +10 mm of storage
N_GROUPS = 3                             # top vs bottom third for the plain-language contrast
EXCLUDE_YEAR_SENS = 2012                 # excluded in one of the D3+ sensitivity runs

# ---- Bootstrap ----
N_BOOT = 5000
BOOT_SEED = 20261017
CI = (2.5, 97.5)

# ---- Specifications: controls added to aws_100cm_mm ----
SPECS = {
    "A: storage only":            [],
    "B: + drought weeks, heat":   [COL_WEEKS_D2, COL_HEAT],
    "C: + longitude (region)":    [COL_WEEKS_D2, COL_HEAT, COL_LON],
    "D: + nccpi_crop":            [COL_WEEKS_D2, COL_HEAT, COL_LON, COL_NCCPI],
}

## Data: Iowa, reported yields, county centroids

In [3]:
ia = df[(df[COL_STATE] == FOCUS_STATE) & (df[COL_CROP].isin(FOCUS_CROPS)) & df[COL_YIELD].notna()].copy()
assert ia[COL_FIPS].map(type).eq(str).all() and ia[COL_FIPS].str.len().eq(5).all(), "fips must be 5-char text"
print("Iowa reported rows by crop:", ia.groupby(COL_CROP).size().to_dict())
print("Rows with NaN yield_anomaly_pct among reported rows:", int(ia[COL_ANOM].isna().sum()))
print("Rows with NaN severity:", int(ia[COL_SEV].isna().sum()))
ia = ia[ia[COL_ANOM].notna()]

# County centroid longitude from the GeoJSON (mean of polygon vertices; good enough for an east-west control)
geo = load_counties()
def centroid_lon(feature):
    g = feature["geometry"]
    polys = g["coordinates"] if g["type"] == "Polygon" else [p for mp in g["coordinates"] for p in mp]
    pts = np.array([pt for ring in polys for pt in ring])
    return pts[:, 0].mean()
lon = pd.DataFrame([(str(f["id"]), centroid_lon(f)) for f in geo["features"] if str(f["id"]).startswith("19")],
                   columns=[COL_FIPS, COL_LON])
ia = ia.merge(lon, on=COL_FIPS, how="left")
assert ia[COL_LON].notna().all(), "some Iowa county has no centroid"

# State-wide storage groups (same tertiles as notebook 02: computed over all 99 counties, not per year)
county_aws = ia.groupby(COL_FIPS)[COL_AWS].first()
assert county_aws.index.nunique() == 99
GROUP_LABELS = ["low", "middle", "high"] if N_GROUPS == 3 else ["low", "high"]
grp = pd.qcut(county_aws, q=N_GROUPS, labels=GROUP_LABELS).astype(str).rename("storage_group")
ia = ia.merge(grp, left_on=COL_FIPS, right_index=True, how="left")
print("Storage-group cutoffs (mm):", county_aws.quantile(np.linspace(0, 1, N_GROUPS + 1)).round(1).tolist())
print("Counties per group:", grp.value_counts().to_dict())

Iowa reported rows by crop: {'CORN': 2470, 'SOYBEANS': 2437}
Rows with NaN yield_anomaly_pct among reported rows: 0
Rows with NaN severity: 0
Storage-group cutoffs (mm): [154.2, 174.1, 182.7, 205.8]
Counties per group: {'middle': 33, 'low': 33, 'high': 33}


## Estimation functions

In [4]:
def ols_slope(d, controls):
    """OLS of yield_anomaly_pct on aws_100cm_mm + controls, within one year.
    Returns the aws coefficient scaled to per +10 mm. NaN if the design is rank deficient."""
    cols = [COL_AWS] + list(controls)
    X = d[cols].to_numpy(dtype=float)
    X = np.column_stack([np.ones(len(d)), X])
    y = d[COL_ANOM].to_numpy(dtype=float)
    if np.linalg.matrix_rank(X) < X.shape[1]:
        return np.nan
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    return beta[1] * AWS_UNIT


def select_years(sub, min_n=MIN_COUNTIES_PER_YEAR):
    """Split years into kept / dropped by the number of drought counties."""
    n = sub.groupby(COL_YEAR)[COL_FIPS].nunique()
    kept = n[n >= min_n]
    dropped = n[n < min_n]
    return kept, dropped


def per_year_slopes(sub, controls):
    """One slope per year (years already filtered)."""
    out = []
    for yr, d in sub.groupby(COL_YEAR):
        out.append({COL_YEAR: yr, "n_counties": d[COL_FIPS].nunique(), "slope_per_10mm": ols_slope(d, controls)})
    return pd.DataFrame(out).set_index(COL_YEAR)


def per_year_contrast(sub):
    """Within each year: mean anomaly of high-storage minus low-storage counties among that year's drought counties."""
    out = []
    for yr, d in sub.groupby(COL_YEAR):
        hi = d.loc[d["storage_group"] == "high", COL_ANOM]
        lo = d.loc[d["storage_group"] == "low", COL_ANOM]
        out.append({COL_YEAR: yr, "n_high": len(hi), "n_low": len(lo),
                    "high_minus_low_pp": (hi.mean() - lo.mean()) if len(hi) and len(lo) else np.nan})
    return pd.DataFrame(out).set_index(COL_YEAR)


def year_bootstrap(values, n_boot=N_BOOT, seed=BOOT_SEED, ci=CI):
    """Resample YEARS with replacement; return (mean, lower, upper, n_years)."""
    v = np.asarray(pd.Series(values).dropna(), dtype=float)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(v), size=(n_boot, len(v)))
    means = v[idx].mean(axis=1)
    lo, hi = np.percentile(means, ci)
    return v.mean(), lo, hi, len(v)


def run_crop(sub_all, crop, sev_threshold, exclude_year=None, label=""):
    """Run all specs + the group contrast for one crop and one drought definition."""
    sub = sub_all[(sub_all[COL_CROP] == crop) & (sub_all[COL_SEV] >= sev_threshold)]
    if exclude_year is not None:
        sub = sub[sub[COL_YEAR] != exclude_year]
    kept, dropped = select_years(sub)
    sub = sub[sub[COL_YEAR].isin(kept.index)]

    rows, slopes = [], {}
    for spec, controls in SPECS.items():
        s = per_year_slopes(sub, controls)
        slopes[spec] = s
        m, lo, hi, n = year_bootstrap(s["slope_per_10mm"])
        rows.append({"crop": crop, "spec": spec, "estimate_per_10mm": m, "ci_low": lo, "ci_high": hi,
                     "n_years": n, "n_county_years": int(s["n_counties"].sum()),
                     "years_positive": int((s["slope_per_10mm"] > 0).sum()),
                     "years_negative": int((s["slope_per_10mm"] < 0).sum()),
                     "excludes_zero": (lo > 0) or (hi < 0)})
    c = per_year_contrast(sub)
    m, lo, hi, n = year_bootstrap(c["high_minus_low_pp"])
    rows.append({"crop": crop, "spec": "high minus low third (pp)", "estimate_per_10mm": m, "ci_low": lo, "ci_high": hi,
                 "n_years": n, "n_county_years": int((c["n_high"] + c["n_low"]).sum()),
                 "years_positive": int((c["high_minus_low_pp"] > 0).sum()),
                 "years_negative": int((c["high_minus_low_pp"] < 0).sum()),
                 "excludes_zero": (lo > 0) or (hi < 0)})
    return {"label": label, "crop": crop, "kept": kept, "dropped": dropped, "sub": sub,
            "results": pd.DataFrame(rows), "slopes": slopes, "contrast": c}


def show_years(res):
    print(f"[{res['label']} / {res['crop']}] years kept: {len(res['kept'])} "
          f"({', '.join(f'{y} (n={n})' for y, n in res['kept'].items())})")
    if len(res["dropped"]):
        print(f"  years dropped (< {MIN_COUNTIES_PER_YEAR} drought counties): "
              f"{', '.join(f'{y} (n={n})' for y, n in res['dropped'].items())}")
    else:
        print("  years dropped: none")


def plot_slopes(res, spec="A: storage only"):
    s = res["slopes"][spec].reset_index()
    r = res["results"].set_index("spec").loc[spec]
    fig = go.Figure()
    fig.add_hrect(y0=r["ci_low"], y1=r["ci_high"], fillcolor="#2b8cbe", opacity=0.15, line_width=0,
                  annotation_text="95% year-bootstrap range", annotation_position="top left")
    fig.add_hline(y=r["estimate_per_10mm"], line_color="#2b8cbe", line_width=2,
                  annotation_text=f"mean {r['estimate_per_10mm']:+.2f} pp / 10 mm", annotation_position="bottom right")
    fig.add_hline(y=0, line_color="grey", line_dash="dot")
    fig.add_trace(go.Scatter(x=s[COL_YEAR], y=s["slope_per_10mm"], mode="markers",
                             marker=dict(size=np.clip(s["n_counties"] / 4, 6, 26), color="#d7301f"),
                             text=[f"{y}: n={n}" for y, n in zip(s[COL_YEAR], s["n_counties"])],
                             hovertemplate="%{text}<br>slope %{y:.2f} pp per 10 mm<extra></extra>",
                             name="per-year slope"))
    fig.update_layout(title=f"{res['crop']} · {res['label']} · {spec}: per-year slope of yield anomaly on storage "
                            f"(marker size = number of drought counties)",
                      xaxis_title="year", yaxis_title="pp of yield anomaly per +10 mm storage", height=420)
    fig.show()

---
## Main analysis: D2+ drought years

### Corn

In [5]:
corn = run_crop(ia, "CORN", MAIN_SEV, label="D2+")
show_years(corn)
display(corn["results"].round(2))

[D2+ / CORN] years kept: 13 (2000 (n=83), 2003 (n=92), 2005 (n=26), 2006 (n=51), 2011 (n=16), 2012 (n=99), 2013 (n=64), 2017 (n=26), 2020 (n=49), 2021 (n=57), 2022 (n=44), 2023 (n=86), 2024 (n=24))
  years dropped (< 15 drought counties): 2002 (n=14), 2007 (n=10), 2014 (n=7), 2018 (n=14)


,crop,spec,estimate_per_10mm,ci_low,ci_high,n_years,n_county_years,years_positive,years_negative,excludes_zero
0,CORN,A: storage only,0.68,-1.81,3.43,13,717,8,5,False
1,CORN,"B: + drought weeks, heat",0.17,-1.61,1.71,13,717,8,5,False
2,CORN,C: + longitude (region),-0.45,-2.16,1.17,13,717,6,7,False
3,CORN,D: + nccpi_crop,-0.32,-1.90,1.23,13,717,6,7,False
4,CORN,high minus low third (pp),-0.24,-4.64,3.45,11,470,7,4,False


In [6]:
print("Per-year slopes (pp of yield anomaly per +10 mm storage), corn, D2+ years:")
tab = pd.concat({spec: s["slope_per_10mm"] for spec, s in corn["slopes"].items()}, axis=1)
tab.insert(0, "n_counties", corn["slopes"]["A: storage only"]["n_counties"])
tab = tab.join(corn["contrast"][["n_high", "n_low", "high_minus_low_pp"]])
display(tab.round(2))
plot_slopes(corn, "A: storage only")
plot_slopes(corn, "C: + longitude (region)")

Per-year slopes (pp of yield anomaly per +10 mm storage), corn, D2+ years:


,n_counties,A: storage only,"B: + drought weeks, heat",C: + longitude (region),D: + nccpi_crop,n_high,n_low,high_minus_low_pp
year,,,,,,,,
2000,83,-3.70,-4.86,-6.23,-5.12,33,25,-9.54
2003,92,-0.37,-0.57,0.52,0.40,28,32,-0.38
2005,26,-7.19,-1.57,-1.45,1.29,0,14,NaN
2006,51,-5.69,-6.41,-3.37,-3.63,23,13,-16.52
2011,16,14.07,3.17,3.04,-0.29,0,8,NaN
2012,99,4.08,3.90,4.99,5.01,33,33,11.02
2013,64,2.01,3.37,3.67,4.81,23,17,2.66
2017,26,1.38,0.47,-1.42,-1.58,5,10,3.38
2020,49,3.25,3.64,-3.65,-3.17,29,4,3.95


### Soybeans

In [7]:
soy = run_crop(ia, "SOYBEANS", MAIN_SEV, label="D2+")
show_years(soy)
display(soy["results"].round(2))

[D2+ / SOYBEANS] years kept: 14 (2000 (n=83), 2003 (n=92), 2005 (n=47), 2006 (n=51), 2011 (n=33), 2012 (n=99), 2013 (n=65), 2017 (n=30), 2018 (n=17), 2020 (n=44), 2021 (n=53), 2022 (n=61), 2023 (n=81), 2024 (n=49))
  years dropped (< 15 drought counties): 2002 (n=14), 2007 (n=10), 2014 (n=7)


,crop,spec,estimate_per_10mm,ci_low,ci_high,n_years,n_county_years,years_positive,years_negative,excludes_zero
0,SOYBEANS,A: storage only,1.11,-0.25,2.40,14,805,9,5,False
1,SOYBEANS,"B: + drought weeks, heat",0.73,-0.55,2.29,14,805,7,7,False
2,SOYBEANS,C: + longitude (region),0.38,-1.18,1.96,14,805,8,6,False
3,SOYBEANS,D: + nccpi_crop,0.29,-1.04,1.62,14,805,8,6,False
4,SOYBEANS,high minus low third (pp),1.90,-1.02,4.72,13,533,7,6,False


In [8]:
print("Per-year slopes (pp of yield anomaly per +10 mm storage), soybeans, D2+ years:")
tab = pd.concat({spec: s["slope_per_10mm"] for spec, s in soy["slopes"].items()}, axis=1)
tab.insert(0, "n_counties", soy["slopes"]["A: storage only"]["n_counties"])
tab = tab.join(soy["contrast"][["n_high", "n_low", "high_minus_low_pp"]])
display(tab.round(2))
plot_slopes(soy, "A: storage only")
plot_slopes(soy, "C: + longitude (region)")

Per-year slopes (pp of yield anomaly per +10 mm storage), soybeans, D2+ years:


,n_counties,A: storage only,"B: + drought weeks, heat",C: + longitude (region),D: + nccpi_crop,n_high,n_low,high_minus_low_pp
year,,,,,,,,
2000,83,-3.81,-2.24,-4.46,-4.33,33,25,-8.82
2003,92,2.62,1.92,1.59,1.76,28,32,6.16
2005,47,-2.21,-3.27,-4.72,-3.63,7,20,-3.04
2006,51,-0.13,-0.42,-0.64,-0.67,23,13,-0.71
2011,33,4.83,-0.79,0.76,0.89,9,10,6.67
2012,99,-0.36,-0.43,2.46,2.12,33,33,-1.15
2013,65,4.56,2.76,4.24,5.05,23,17,9.42
2017,30,3.55,1.69,-0.85,-0.78,6,11,10.60
2018,17,4.64,8.39,6.54,4.27,0,14,NaN


### Main results, side by side

In [9]:
main = pd.concat([corn["results"], soy["results"]], ignore_index=True)
main_view = main.copy()
main_view["estimate [95% range]"] = main_view.apply(
    lambda r: f"{r['estimate_per_10mm']:+.2f} [{r['ci_low']:+.2f}, {r['ci_high']:+.2f}]", axis=1)
main_view["years +/-"] = main_view["years_positive"].astype(str) + " / " + main_view["years_negative"].astype(str)
display(main_view.set_index(["crop", "spec"])[["estimate [95% range]", "n_years", "n_county_years", "years +/-", "excludes_zero"]])
print("Units: specs A-D are percentage points of yield anomaly per +10 mm of aws_100cm_mm.")
print("       'high minus low third' is percentage points of yield anomaly (high-storage mean minus low-storage mean).")

estimate [95% range]  n_years  n_county_years years +/-  excludes_zero
crop     spec                                                                                             
CORN     A: storage only            +0.68 [-1.81, +3.43]       13             717     8 / 5          False
         B: + drought weeks, heat   +0.17 [-1.61, +1.71]       13             717     8 / 5          False
         C: + longitude (region)    -0.45 [-2.16, +1.17]       13             717     6 / 7          False
         D: + nccpi_crop            -0.32 [-1.90, +1.23]       13             717     6 / 7          False
         high minus low third (pp)  -0.24 [-4.64, +3.45]       11             470     7 / 4          False
SOYBEANS A: storage only            +1.11 [-0.25, +2.40]       14             805     9 / 5          False
         B: + drought weeks, heat   +0.73 [-0.55, +2.29]       14             805     7 / 7          False
         C: + longitude (region)    +0.38 [-1.18, +1.96]       14             805     8 / 6          False
         D: + nccpi_crop            +0.29 [-1.04, +1.62]       14             805     8 / 6          False
         high minus low third (pp)  +1.90 [-1.02, +4.72]       13             533     7 / 6          False

Units: specs A-D are percentage points of yield anomaly per +10 mm of aws_100cm_mm.
       'high minus low third' is percentage points of yield anomaly (high-storage mean minus low-storage mean).


---
## Sensitivity: D3+ drought years, specification C (with and without 2012)

Specification C = storage + drought weeks + heat days + longitude.

In [10]:
SENS_SPEC = "C: + longitude (region)"
sens_runs = []
for crop in FOCUS_CROPS:
    for excl in [None, EXCLUDE_YEAR_SENS]:
        lab = "D3+" + ("" if excl is None else f" excl. {excl}")
        r = run_crop(ia, crop, SENS_SEV, exclude_year=excl, label=lab)
        show_years(r)
        sens_runs.append(r)
        print()

sens = pd.concat([r["results"].assign(definition=r["label"]) for r in sens_runs], ignore_index=True)
sens = sens[sens["spec"].isin([SENS_SPEC, "high minus low third (pp)"])]
sens_view = sens.copy()
sens_view["estimate [95% range]"] = sens_view.apply(
    lambda r: f"{r['estimate_per_10mm']:+.2f} [{r['ci_low']:+.2f}, {r['ci_high']:+.2f}]", axis=1)
sens_view["years +/-"] = sens_view["years_positive"].astype(str) + " / " + sens_view["years_negative"].astype(str)
display(sens_view.set_index(["crop", "definition", "spec"])[["estimate [95% range]", "n_years", "n_county_years", "years +/-", "excludes_zero"]])

[D3+ / CORN] years kept: 7 (2003 (n=31), 2005 (n=15), 2012 (n=92), 2020 (n=20), 2021 (n=25), 2022 (n=21), 2023 (n=41))
  years dropped (< 15 drought counties): 2000 (n=14), 2013 (n=6), 2017 (n=9), 2018 (n=3)

[D3+ excl. 2012 / CORN] years kept: 6 (2003 (n=31), 2005 (n=15), 2020 (n=20), 2021 (n=25), 2022 (n=21), 2023 (n=41))
  years dropped (< 15 drought counties): 2000 (n=14), 2013 (n=6), 2017 (n=9), 2018 (n=3)

[D3+ / SOYBEANS] years kept: 8 (2000 (n=26), 2003 (n=48), 2005 (n=15), 2012 (n=93), 2020 (n=24), 2021 (n=23), 2022 (n=22), 2023 (n=44))
  years dropped (< 15 drought counties): 2013 (n=6), 2017 (n=10), 2018 (n=6)



[D3+ excl. 2012 / SOYBEANS] years kept: 7 (2000 (n=26), 2003 (n=48), 2005 (n=15), 2020 (n=24), 2021 (n=23), 2022 (n=22), 2023 (n=44))
  years dropped (< 15 drought counties): 2013 (n=6), 2017 (n=10), 2018 (n=6)



estimate [95% range]  n_years  n_county_years years +/-  excludes_zero
crop     definition     spec                                                                                             
CORN     D3+            C: + longitude (region)    +1.41 [-0.29, +3.17]        7             245     5 / 2          False
                        high minus low third (pp)  +0.53 [-2.40, +4.83]        6             167     2 / 4          False
         D3+ excl. 2012 C: + longitude (region)    +0.85 [-0.70, +2.73]        6             153     4 / 2          False
                        high minus low third (pp)  -1.51 [-2.85, -0.35]        5             105     1 / 4           True
SOYBEANS D3+            C: + longitude (region)    +0.84 [-0.89, +2.22]        8             295     6 / 2          False
                        high minus low third (pp)  -0.68 [-4.88, +2.39]        7             204     3 / 4          False
         D3+ excl. 2012 C: + longitude (region)    +0.58 [-1.35, +2.08]        7             202     5 / 2          False
                        high minus low third (pp)  -0.68 [-5.67, +2.90]        6             141     3 / 3          False

In [11]:
# Per-year slopes behind the D3+ sensitivity (spec C), so the small number of years is visible
for r in sens_runs:
    if "excl" in r["label"]:
        continue
    s = r["slopes"][SENS_SPEC].copy()
    s["is_2012"] = s.index == EXCLUDE_YEAR_SENS
    print(f"{r['crop']} · {r['label']} · {SENS_SPEC}")
    display(s.round(2))

CORN · D3+ · C: + longitude (region)


,n_counties,slope_per_10mm,is_2012
year,,,
2003,31,1.03,False
2005,15,4.97,False
2012,92,4.80,True
2020,20,-2.22,False
2021,25,0.61,False
2022,21,0.78,False
2023,41,-0.11,False


SOYBEANS · D3+ · C: + longitude (region)


,n_counties,slope_per_10mm,is_2012
year,,,
2000,26,-4.32,False
2003,48,2.64,False
2005,15,2.18,False
2012,93,2.68,True
2020,24,-0.69,False
2021,23,0.62,False
2022,22,2.90,False
2023,44,0.71,False


---
## Combined comparison: D2+ main vs D3+ sensitivity (spec C)

In [12]:
combo = pd.concat([
    main[main["spec"] == SENS_SPEC].assign(definition="D2+ (main)"),
    sens[sens["spec"] == SENS_SPEC],
], ignore_index=True)
combo["estimate [95% range]"] = combo.apply(
    lambda r: f"{r['estimate_per_10mm']:+.2f} [{r['ci_low']:+.2f}, {r['ci_high']:+.2f}]", axis=1)
display(combo.set_index(["crop", "definition"])[["estimate [95% range]", "n_years", "n_county_years", "excludes_zero"]])

estimate [95% range]  n_years  n_county_years  excludes_zero
crop     definition                                                                  
CORN     D2+ (main)      -0.45 [-2.16, +1.17]       13             717          False
SOYBEANS D2+ (main)      +0.38 [-1.18, +1.96]       14             805          False
CORN     D3+             +1.41 [-0.29, +3.17]        7             245          False
         D3+ excl. 2012  +0.85 [-0.70, +2.73]        6             153          False
SOYBEANS D3+             +0.84 [-0.89, +2.22]        8             295          False
         D3+ excl. 2012  +0.58 [-1.35, +2.08]        7             202          False

---
## Reading the results (plain language)

Units: percentage points (pp) of yield anomaly per **+10 mm** of soil water storage, averaged over drought years, with a 95% range from resampling years. A positive number means counties with more storage lost less yield in drought years. For scale, 10 mm is about one standard deviation of storage across Iowa counties, and the whole state spans only about 50 mm.

**Corn (D2+, 13 drought years, 717 county-years).** No specification's range excludes zero.
- A (storage only): +0.68 [−1.81, +3.43]. 8 years positive, 5 negative.
- B (+ drought weeks, heat): +0.17 [−1.61, +1.71].
- C (+ longitude): −0.45 [−2.16, +1.17]. 6 years positive, 7 negative.
- D (+ nccpi_crop): −0.32 [−1.90, +1.23].
- High minus low third: −0.24 pp [−4.64, +3.45] over 11 years (2 years had no high-storage county in D2+).

The raw estimate is small and positive, shrinks once drought intensity and heat are controlled, and crosses to slightly negative once longitude is added. The 2012 statewide drought on its own shows a storage benefit of about +4 to +5 pp per 10 mm in every specification, but 2000, 2006 and 2023 point the other way, and the year-to-year spread is far larger than the mean.

**Soybeans (D2+, 14 drought years, 805 county-years).** No specification's range excludes zero.
- A (storage only): +1.11 [−0.25, +2.40]. 9 years positive, 5 negative.
- B (+ drought weeks, heat): +0.73 [−0.55, +2.29].
- C (+ longitude): +0.38 [−1.18, +1.96].
- D (+ nccpi_crop): +0.29 [−1.04, +1.62].
- High minus low third: +1.90 pp [−1.02, +4.72] over 13 years.

Soybeans show a consistently positive but imprecise association. The raw estimate comes closest to excluding zero; each added control roughly halves it, and the longitude and soil-quality controls together take it from +1.1 to +0.3. That pattern is what you would expect if part of the raw association is really "west vs east" and "better soil" rather than storage itself.

**D3+ sensitivity (spec C).** With only 6 to 8 usable years the ranges are wide and nothing excludes zero: corn +1.41 [−0.29, +3.17] with 2012, +0.85 [−0.70, +2.73] without; soybeans +0.84 [−0.89, +2.22] with 2012, +0.58 [−1.35, +2.08] without. The D3+ estimates are more positive than D2+, so the strongest droughts hint at a storage benefit, but the evidence is thin and 2012 alone is a third of the D3+ county-years. The one range that excludes zero (corn, high minus low third, D3+ without 2012: −1.51 [−2.85, −0.35]) rests on 5 years and 105 county-years and points the opposite way from the regression in the same subset; it should be treated as noise, not a finding.

**Bottom line.** In this in-sample analysis, the data do not show a clear association between soil water storage and drought-year yield loss for Iowa corn, and only a weak, uncertain positive association for soybeans that fades as region and soil quality are controlled. "No clear effect" is the honest reading at this stage. These are associations across counties with fixed soil, not the measured effect of any conservation program, and they have not yet been tested on held-out years.
